# scEvoNet — full workflow (PBMC3k-derived demo)

This notebook uses **`scevonet.pbmc_demo.build_pbmc_demo_bundle`**: Scanpy’s PBMC3k is downloaded once (~6 MB), then we subset cells/genes, add noise, and derive synthetic cluster labels (including a **Rare** type).

**Install:** `pip install scevonet scanpy` (or `pip install -e ".[dev]"` from a clone).


## 1. Imports and demo data


In [ ]:
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

from scevonet import (
    EvoManager,
    Sample,
    SampleConfig,
    bootstrap_importance_stability,
    enrich_genes,
    finish_matplotlib_figure,
    leave_batch_out_auc,
    permutation_importance_null,
)
from scevonet.programs import classify_transition_genes, cluster_mean_expression
from scevonet.pbmc_demo import build_pbmc_demo_bundle
from scevonet.viz import draw_network

bundle = build_pbmc_demo_bundle(seed=42)
mat_a = bundle["df_a"]
labels_a = bundle["labels_a"]
mat_b = bundle["df_b"]
labels_b = bundle["labels_b"]
mat_c = bundle["df_c"]
labels_c = bundle["labels_c"]
batches = bundle["batches"]
ca, cb = bundle["cluster_a"], bundle["cluster_b"]
genes_tr = bundle["genes_for_transition"]

print(mat_a.shape, "genes:", len(mat_a.columns))
print("Cell types:", sorted(set(labels_a)))


## 2. Train `Sample` models and run `EvoManager`

Three “datasets” share the same gene columns (bootstrap resamples + noise).


In [ ]:
cfg = SampleConfig(
    top_features_limit=200,
    n_estimators=80,
    early_stopping_rounds=8,
    random_state=42,
)

s1 = Sample(mat_a, labels_a, config=cfg)
s2 = Sample(mat_b, labels_b, config=cfg)
s3 = Sample(mat_c, labels_c, config=cfg)

em = EvoManager(s1, s2, s3, network_top_genes=40)

list(em.predictions.keys())[:5], em.cell_types_similarity.shape


## 3. Similarity matrix and gene–cell network edges


In [ ]:
print(em.cell_types_similarity.iloc[:8, :8].to_string())
print()
print(em.network.head(12).to_string())


## 4. Plot a small subgraph (`draw_network`)

We take the top edges from the merged long-format network.


In [ ]:
sub_edges = em.network.sort_values("Importance", ascending=False).head(35)
G = nx.Graph()
for _, r in sub_edges.iterrows():
    G.add_edge(r["Cell_type"], r["Gene"], importance=float(r["Importance"]))

fig = draw_network(G, gene=True, title="Top gene–cell edges (PBMC demo)")
finish_matplotlib_figure(fig)


## 5. Directionality — pseudobulk contrast between two clusters


In [ ]:
mu = cluster_mean_expression(mat_a, labels_a, ca)
transition = classify_transition_genes(
    mat_a,
    labels_a,
    genes_tr,
    ca,
    cb,
    log2_fc_strong=0.12,
    rank_high=0.52,
    rank_low=0.38,
)
transition.head(10)


## 6. Validation — permutation null and leave-batch-out AUC


In [ ]:
perm = permutation_importance_null(
    mat_a,
    labels_a,
    sorted(set(labels_a))[0],
    config=cfg,
    n_perm=8,
    random_seed=1,
)
perm["p_value"], perm["observed"]

lobo = leave_batch_out_auc(mat_a, labels_a, batches, sorted(set(labels_a))[0], config=cfg)
lobo


## 7. Bootstrap stability of feature rankings (optional; heavier)


In [ ]:
stab_tab, ref_rank = bootstrap_importance_stability(
    mat_a,
    labels_a,
    sorted(set(labels_a))[0],
    config=cfg,
    n_bootstrap=6,
    top_k=40,
    random_seed=0,
)
stab_tab.head()


## 8. Shortest-path subnetwork between two cell-type nodes


In [ ]:
try:
    subnet = em.generate_cell_type_network(
        ca,
        cb,
        number_of_shortest_paths=50,
        minimal_number_of_nodes=3,
        closest_clusters=12,
    )
    print("Subgraph nodes:", subnet.number_of_nodes(), "edges:", subnet.number_of_edges())
except Exception as e:
    print("Subnetwork step skipped:", e)


## 9. Optional — Enrichr / GO (`gseapy`)

Requires `pip install 'scevonet[enrichment]'` and network access.


In [ ]:
try:
    top_genes = (
        s1.models_top_features[sorted(s1.models_top_features.keys())[0]]["gene"]
        .head(25)
        .astype(str)
        .tolist()
    )
    enr = enrich_genes(top_genes[:20], organism="human", no_plot=True)
    enr.head(5)
except ImportError as e:
    print("Install enrichment extra:", e)
except Exception as e:
    print("Enrichment run skipped:", e)


## 10. Optional — `sample_from_adata`

Build a minimal `AnnData` from the demo matrix (requires `pip install 'scevonet[anndata]'`).


In [ ]:
try:
    import anndata as ad

    adata = ad.AnnData(X=mat_a.to_numpy(), obs=pd.DataFrame(index=mat_a.index), var=pd.DataFrame(index=mat_a.columns))
    adata.obs["cluster"] = labels_a
    from scevonet import sample_from_adata

    s_ada = sample_from_adata(adata, "cluster")
    len(s_ada.models)
except ImportError as e:
    print("Install anndata extra:", e)
